# 01 - The data, and why it is not circular

Everything here is synthetic. The interesting question is not "is it real" --
it is not -- but "does it beg the question it is supposed to answer."

Most synthetic fraud datasets do. The author labels rows as fraud, injects
features that mark them, trains a model that rediscovers the labels, and
reports a precision that measures nothing but the author's imagination.

This generator avoids that in one specific way: **false positives are never
injected. They emerge.**

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np, warnings
warnings.filterwarnings('ignore')
DATA = '../data300k'      # the working set
DATA100K = '../data'      # the calibrated baseline the pitch quotes

In [2]:
import json
stats100 = json.load(open(f'{DATA100K}/stats.json'))['stats']
stats300 = json.load(open(f'{DATA}/stats.json'))['stats']

for name, s in [('100k baseline', stats100), ('300k working set', stats300)]:
    print(f'--- {name} ---')
    print(f"  payments            {s['payments']:,}")
    print(f"  blocked by the stack{s['blocked']:>8,}  ({s['block_rate']:.2%})")
    print(f"  of those, GOOD      {s['blocked_that_were_good']:>8,}  ({s['fp_share_of_blocked_pile']:.1%} of the pile)")
    print(f"  revenue refused     Rs {s['revenue_wrongly_blocked_inr']/1e7:>8,.2f} cr")
    print(f"  fraud caught        Rs {s['fraud_correctly_blocked_inr']/1e7:>8,.2f} cr")
    print(f"  value ratio         {s['fp_to_tp_value_ratio']}x")

--- 100k baseline ---
  payments            100,000
  blocked by the stack   2,817  (2.82%)
  of those, GOOD         2,128  (75.5% of the pile)
  revenue refused     Rs    14.79 cr
  fraud caught        Rs     3.90 cr
  value ratio         3.79x
--- 300k working set ---
  payments            300,000
  blocked by the stack   8,265  (2.76%)
  of those, GOOD         6,346  (76.8% of the pile)
  revenue refused     Rs    44.25 cr
  fraud caught        Rs    10.81 cr
  value ratio         4.09x


## Two stages that never talk to each other

**Stage 1** invents the true world. Every customer carries a hidden persona and
every order carries a true counterfactual outcome -- what *would* have happened
if it had been allowed through.

**Stage 2** runs a merchant scorecard that never sees persona or outcome. It
sees a dozen observable signals: new device, address mismatch, odd hour,
unusual basket, high-RTO pincode, disposable email.

Honest-but-atypical customers emit the *same observable signals* as fraudsters.
So the scorecard blocks some of them. That mismatch is the false-positive
population. No parameter anywhere sets a false-positive rate.

In [3]:
from core.feature_store import FeatureStore, LOCAL_FEATURES, NETWORK_FEATURES
from core.truth import TruthVault

store, vault = FeatureStore.load(DATA), TruthVault(DATA)
cases = store.split('holdout')
y = vault.labels(store.payment_ids(cases))    # 1 = the block was correct

print(f'{len(LOCAL_FEATURES)} local features the merchant can see:')
print('   ', ', '.join(LOCAL_FEATURES))
print(f'\n{len(NETWORK_FEATURES)} network features only the platform can see:')
print('   ', ', '.join(NETWORK_FEATURES))
print(f'\nholdout: {len(cases)} appealed cases, {(y==0).sum()} of them good ({(y==0).mean():.1%})')

14 local features the merchant can see:
    f_device_is_new, f_device_account_fanout, f_address_mismatch, f_orders_last_24h, f_amount_z, f_pincode_rto_propensity, f_is_night, f_thin_file_flag, f_disposable_email, f_international, f_merchant_prior_rto, f_is_cod, risk_score, amount

8 network features only the platform can see:
    network_orders_prior, network_merchants_prior, network_tenure_days, network_clean_rate, network_disputes_prior, network_rto_prior, network_device_fanout, network_instrument_merchants

holdout: 1775 appealed cases, 1435 of them good (80.8%)


## The overlap is the whole problem

If the fraud tells separated cleanly, this project would be trivial. Here is
what a merchant actually sees when it looks at a good customer versus a bad one
in its blocked pile.

In [4]:
import numpy as np
X = store.as_matrix(cases, ('local',))
names = LOCAL_FEATURES
print(f'{"signal":<28}{"good (FP)":>12}{"bad (TP)":>12}   overlap')
print('-'*70)
for j, n in enumerate(names):
    if n in ('amount', 'risk_score', 'f_amount_z'): continue
    g, b = X[y==0, j].mean(), X[y==1, j].mean()
    bar = '#' * int(30 * min(g, b) / max(g, b, 1e-9))
    print(f'{n:<28}{g:>12.3f}{b:>12.3f}   {bar}')

signal                         good (FP)    bad (TP)   overlap
----------------------------------------------------------------------
f_device_is_new                    0.553       0.650   #########################
f_device_account_fanout            1.329       1.971   ####################
f_address_mismatch                 0.558       0.618   ###########################
f_orders_last_24h                  0.093       0.909   ###
f_pincode_rto_propensity           1.128       1.102   #############################
f_is_night                         0.549       0.559   #############################
f_thin_file_flag                   0.050       0.288   #####
f_disposable_email                 0.291       0.476   ##################
f_international                    0.202       0.209   #############################
f_merchant_prior_rto               1.600       0.626   ###########
f_is_cod                           0.314       0.668   ##############


Read the overlap column. `f_disposable_email` fires for honest people who
protect their inbox. `f_device_is_new` fires for anyone who bought a phone.
`f_pincode_rto_propensity` is high across Tier-2 and Tier-3 India -- which is
to say, a risk stack tuned on regional RTO rates refuses customers in Patna and
Guwahati more often than customers in Bengaluru.

Each of those is individually defensible. In aggregate they are a merchant
quietly withdrawing from the fastest-growing part of its market.

In [5]:
# Which stated block reasons produce the most wrongful declines?
from collections import defaultdict
agg = defaultdict(lambda: [0, 0, 0.0])
for e, bad in zip(cases, y):
    a = agg[e.meta['block_reason']]
    a[0] += 1
    a[1] += (bad == 0)
    a[2] += e.amount_inr * (bad == 0)

print(f'{"block reason":<24}{"blocked":>9}{"wrongly":>9}{"FP rate":>9}{"Rs refused":>14}')
print('-'*68)
for r, (n, fp, amt) in sorted(agg.items(), key=lambda kv: -kv[1][2]):
    print(f'{r:<24}{n:>9}{fp:>9}{fp/n:>9.0%}{amt/1e5:>12,.1f} L')

block reason              blocked  wrongly  FP rate    Rs refused
--------------------------------------------------------------------
amount_anomaly                307      246      80%       180.1 L
device_reputation             333      255      77%       145.5 L
address_mismatch              275      215      78%       128.6 L
rto_history                   398      364      91%       128.1 L
geo_risk                      206      172      83%        81.7 L
instrument_risk               214      167      78%        63.6 L
thin_file_high_value           25       14      56%        15.1 L
velocity_burst                 17        2      12%         0.2 L


## Calibration, and where the docs are stale

The generator was tuned against public anchors. Two of them:

- Block rate ~2.7% (ClearSale, US domestic, Q3 2023) - we sit at ~2.8%.
- 30-70% of merchant-declined orders are actually good (Signifyd) - we sit at
  ~76%, *above* the published band. Stated, not hidden.

**Watch out:** `DATA_CARD.md` and `IDEA.md` were written against an older
version of this generator. Their row counts and AUC tables no longer match what
the code produces. `../README.md` has the full audit.

In [6]:
card_claims = {'appeal_queue rows': 2722, 'disputes': 2693, 'refunds': 5762, 'holdout blocked': 615}
actual = {'appeal_queue rows': stats100['blocked'], 'disputes': stats100['disputes'],
          'refunds': stats100['refunds_rto'], 'holdout blocked': stats100['holdout_blocked']}
print(f'{"":<22}{"DATA_CARD says":>16}{"actually":>12}')
for k in card_claims:
    flag = '' if card_claims[k] == actual[k] else '   <-- stale'
    print(f'{k:<22}{card_claims[k]:>16,}{actual[k]:>12,}{flag}')

                        DATA_CARD says    actually
appeal_queue rows                2,722       2,817   <-- stale
disputes                         2,693       2,616   <-- stale
refunds                          5,762       5,844   <-- stale
holdout blocked                    615         622   <-- stale
